In [ ]:
# Import Libraries
import numpy as np
import pandas as pd
from math import sqrt
import matplotlib.pyplot as plt
from matplotlib import rcParams
import time
from datetime import datetime
import itertools
from collections import defaultdict

# PyTorch and PyTorch Geometric
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv, global_mean_pool, BatchNorm
from torch_geometric.data import Data, DataLoader
from torch_geometric.utils import dense_to_sparse

# Sklearn
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Suppress warnings
import warnings
warnings.filterwarnings('ignore')

print(f"PyTorch version: {torch.__version__}")
print(f"GCN Grid Search started at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

# HELPER FUNCTIONS

In [ ]:
def series_to_supervised(data, n_in=1, n_out=1, dropnan=True):
    """Convert time series data to supervised learning format."""
    n_vars = 1 if type(data) is list else data.shape[1]
    df = pd.DataFrame(data)
    cols, names = list(), list()
    
    # Input sequence (t-n, ... t-1)
    for i in range(n_in, 0, -1):
        cols.append(df.shift(i))
        names += [('var%d(t-%d)' % (j+1, i)) for j in range(n_vars)]
    
    # Forecast sequence (t, t+1, ... t+n)
    for i in range(0, n_out):
        cols.append(df.shift(-i))
        if i == 0:
            names += [('var%d(t)' % (j+1)) for j in range(n_vars)]
        else:
            names += [('var%d(t+%d)' % (j+1, i)) for j in range(n_vars)]
    
    agg = pd.concat(cols, axis=1)
    agg.columns = names
    
    if dropnan:
        agg.dropna(inplace=True)
    return agg

In [ ]:
def create_feature_graph(data, correlation_threshold=0.3):
    """Create graph structure based on feature correlations."""
    # Calculate correlation matrix
    corr_matrix = np.corrcoef(data.T)
    
    # Create adjacency matrix based on correlation threshold
    adj_matrix = (np.abs(corr_matrix) > correlation_threshold).astype(float)
    np.fill_diagonal(adj_matrix, 1)  # Self-connections
    
    # Convert to edge list
    edge_index, edge_weights = dense_to_sparse(torch.tensor(adj_matrix, dtype=torch.float))
    
    return edge_index, edge_weights, corr_matrix

In [ ]:
def create_graph_data_list(X, Y, window_size=24):
    """Convert time series data to graph format."""
    graph_data_list = []
    
    # Create base graph structure from feature correlations
    sample_data = X[0].reshape(-1, X.shape[-1])
    edge_index, edge_weights, _ = create_feature_graph(sample_data)
    
    for i in range(len(X)):
        # Node features: each feature becomes a node with temporal values
        node_features = X[i].T  # [features, timesteps]
        
        # Create graph data object
        data = Data(
            x=torch.tensor(node_features, dtype=torch.float32),
            edge_index=edge_index.long(),
            y=torch.tensor([Y[i]], dtype=torch.float32)
        )
        graph_data_list.append(data)
    
    return graph_data_list

# LOAD DATASET

In [ ]:
print("Loading dataset...")
dataset = pd.read_csv("eMalahleniIM.csv", sep=';', header=0, index_col=0)
values = dataset.values
print(f"Dataset shape: {dataset.shape}")

# DATA PREPROCESSING

In [ ]:
print("Preprocessing data...")

# Ensure all data is float
values = values.astype('float32')

In [ ]:
# Normalize features
scaler = MinMaxScaler(feature_range=(0, 1))
scaled = scaler.fit_transform(values)

# GCN MODEL DEFINITION

In [ ]:
class GCNModel(nn.Module):
    """Graph Convolutional Network for air pollution prediction."""
    
    def __init__(self, n_features, hidden_dim=64, n_layers=3, dropout=0.2):
        super(GCNModel, self).__init__()
        
        self.n_layers = n_layers
        self.dropout = dropout
        
        # GCN layers
        self.convs = nn.ModuleList()
        self.batch_norms = nn.ModuleList()
        
        # Input layer
        self.convs.append(GCNConv(n_features, hidden_dim))
        self.batch_norms.append(BatchNorm(hidden_dim))
        
        # Hidden layers
        for _ in range(n_layers - 2):
            self.convs.append(GCNConv(hidden_dim, hidden_dim))
            self.batch_norms.append(BatchNorm(hidden_dim))
        
        # Output layer
        self.convs.append(GCNConv(hidden_dim, hidden_dim))
        self.batch_norms.append(BatchNorm(hidden_dim))
        
        # Final prediction layers
        self.predictor = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, 1),
            nn.Sigmoid()
        )
    
    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch
        
        # GCN layers with residual connections
        for i in range(self.n_layers):
            x_residual = x if i > 0 and x.size(-1) == self.convs[i].out_channels else None
            
            x = self.convs[i](x, edge_index)
            x = self.batch_norms[i](x)
            x = F.relu(x)
            x = F.dropout(x, p=self.dropout, training=self.training)
            
            # Residual connection
            if x_residual is not None:
                x = x + x_residual
        
        # Global pooling to get graph-level representation
        x = global_mean_pool(x, batch)
        
        # Final prediction
        out = self.predictor(x)
        
        return out.squeeze()

# TRAINING FUNCTIONS

In [ ]:
def train_epoch(model, train_loader, optimizer, criterion, device):
    """Train for one epoch."""
    model.train()
    total_loss = 0
    
    for batch in train_loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        
        out = model(batch)
        loss = criterion(out, batch.y)
        
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    
    return total_loss / len(train_loader)

In [ ]:
def evaluate_model(model, data_loader, criterion, device):
    """Evaluate the model."""
    model.eval()
    total_loss = 0
    predictions = []
    targets = []
    
    with torch.no_grad():
        for batch in data_loader:
            batch = batch.to(device)
            out = model(batch)
            loss = criterion(out, batch.y)
            
            total_loss += loss.item()
            predictions.extend(out.cpu().numpy())
            targets.extend(batch.y.cpu().numpy())
    
    return total_loss / len(data_loader), np.array(predictions), np.array(targets)

In [ ]:
def train_and_evaluate_model(params, train_data, val_data, n_features, device, max_epochs=50):
    """Train and evaluate a single model configuration."""
    
    # Create data loaders
    train_loader = DataLoader(train_data, batch_size=params['batch_size'], shuffle=True)
    val_loader = DataLoader(val_data, batch_size=params['batch_size'], shuffle=False)
    
    # Initialize model
    model = GCNModel(
        n_features=n_features,
        hidden_dim=params['hidden_dim'],
        n_layers=params['n_layers'],
        dropout=params['dropout']
    ).to(device)
    
    # Loss function and optimizer
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=params['learning_rate'], weight_decay=1e-5)
    
    # Training loop
    best_val_loss = float('inf')
    patience = 10
    patience_counter = 0
    
    for epoch in range(max_epochs):
        # Train
        train_loss = train_epoch(model, train_loader, optimizer, criterion, device)
        
        # Validate
        val_loss, _, _ = evaluate_model(model, val_loader, criterion, device)
        
        # Early stopping
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
        else:
            patience_counter += 1
        
        if patience_counter >= patience:
            break
    
    return best_val_loss

# GRID SEARCH PARAMETERS

In [ ]:
print("Setting up grid search parameters...")

# Define parameter grid
param_grid = {
    'window_size': [12, 24, 48],
    'hidden_dim': [32, 64, 128],
    'n_layers': [2, 3, 4],
    'dropout': [0.1, 0.2, 0.3],
    'learning_rate': [0.001, 0.0005, 0.0001],
    'batch_size': [16, 32, 64]
}

print("Parameter combinations to test:")
for param, values in param_grid.items():
    print(f"  {param}: {values}")

In [ ]:
# Calculate total combinations
total_combinations = 1
for values in param_grid.values():
    total_combinations *= len(values)
print(f"\nTotal combinations: {total_combinations}")

# GRID SEARCH EXECUTION

In [ ]:
# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

In [ ]:
# Storage for results
results = []
best_score = float('inf')
best_params = None

print(f"\nStarting grid search...")
start_time = time.time()

In [ ]:
combination_count = 0

for window_size in param_grid['window_size']:
    print(f"\n--- Testing window_size: {window_size} ---")
    
    # Create windowed data for current window size
    X_windows = []
    Y_windows = []
    
    for i in range(len(scaled) - window_size):
        X_windows.append(scaled[i:i+window_size])
        Y_windows.append(scaled[i+window_size, 0])  # Predict PM2.5
    
    X = np.array(X_windows)
    Y = np.array(Y_windows)
    n_features = scaled.shape[1]
    
    # Convert to graph data
    graph_data = create_graph_data_list(X, Y, window_size)
    
    # Split data
    n_samples = len(graph_data)
    indices = list(range(n_samples))
    train_val_idx, _ = train_test_split(indices, test_size=0.2, random_state=42)
    train_idx, val_idx = train_test_split(train_val_idx, test_size=0.25, random_state=42)
    
    train_data = [graph_data[i] for i in train_idx]
    val_data = [graph_data[i] for i in val_idx]
    
    # Test other parameters with this window size
    other_params = {k: v for k, v in param_grid.items() if k != 'window_size'}
    param_combinations = list(itertools.product(*other_params.values()))
    
    for combination in param_combinations:
        combination_count += 1
        
        # Create parameter dictionary
        params = dict(zip(other_params.keys(), combination))
        params['window_size'] = window_size
        
        print(f"Testing combination {combination_count}/{total_combinations}: {params}")
        
        try:
            # Train and evaluate
            val_loss = train_and_evaluate_model(params, train_data, val_data, window_size, device)
            
            # Store results
            result = params.copy()
            result['val_loss'] = val_loss
            result['r2_score'] = 1 - val_loss  # Approximate R2 from MSE loss
            results.append(result)
            
            # Update best parameters
            if val_loss < best_score:
                best_score = val_loss
                best_params = params.copy()
                print(f"  New best! Val Loss: {val_loss:.6f}")
            else:
                print(f"  Val Loss: {val_loss:.6f}")
                
        except Exception as e:
            print(f"  Error: {e}")
            continue

total_time = time.time() - start_time
print(f"\nGrid search completed in {total_time:.2f} seconds")

# RESULTS ANALYSIS

In [ ]:
print("\n" + "="*60)
print("GRID SEARCH RESULTS")
print("="*60)

In [ ]:
# Convert results to DataFrame
results_df = pd.DataFrame(results)
results_df = results_df.sort_values('val_loss')

print(f"\nBest Parameters:")
print(f"Val Loss: {best_score:.6f}")
for param, value in best_params.items():
    print(f"  {param}: {value}")

print(f"\nTop 10 Results:")
print(results_df.head(10)[['window_size', 'hidden_dim', 'n_layers', 'dropout', 
                          'learning_rate', 'batch_size', 'val_loss']].to_string(index=False))


# VISUALIZATIONS

In [ ]:
print("\nCreating visualizations...")

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.ravel()

# 1. Window Size vs Performance
window_size_results = results_df.groupby('window_size')['val_loss'].agg(['mean', 'std']).reset_index()
axes[0].errorbar(window_size_results['window_size'], window_size_results['mean'], 
                yerr=window_size_results['std'], marker='o', linewidth=2)
axes[0].set_xlabel('Window Size', fontweight='bold')
axes[0].set_ylabel('Validation Loss', fontweight='bold')
axes[0].set_title('Window Size vs Performance', fontweight='bold')
axes[0].grid(True, alpha=0.3)

In [ ]:
# 2. Hidden Dimensions vs Performance
hidden_dim_results = results_df.groupby('hidden_dim')['val_loss'].agg(['mean', 'std']).reset_index()
axes[1].errorbar(hidden_dim_results['hidden_dim'], hidden_dim_results['mean'], 
                yerr=hidden_dim_results['std'], marker='o', linewidth=2)
axes[1].set_xlabel('Hidden Dimensions', fontweight='bold')
axes[1].set_ylabel('Validation Loss', fontweight='bold')
axes[1].set_title('Hidden Dimensions vs Performance', fontweight='bold')
axes[1].grid(True, alpha=0.3)

In [ ]:
# 3. Number of Layers vs Performance
layers_results = results_df.groupby('n_layers')['val_loss'].agg(['mean', 'std']).reset_index()
axes[2].errorbar(layers_results['n_layers'], layers_results['mean'], 
                yerr=layers_results['std'], marker='o', linewidth=2)
axes[2].set_xlabel('Number of Layers', fontweight='bold')
axes[2].set_ylabel('Validation Loss', fontweight='bold')
axes[2].set_title('Number of Layers vs Performance', fontweight='bold')
axes[2].grid(True, alpha=0.3)

In [ ]:
# 4. Dropout vs Performance
dropout_results = results_df.groupby('dropout')['val_loss'].agg(['mean', 'std']).reset_index()
axes[3].errorbar(dropout_results['dropout'], dropout_results['mean'], 
                yerr=dropout_results['std'], marker='o', linewidth=2)
axes[3].set_xlabel('Dropout Rate', fontweight='bold')
axes[3].set_ylabel('Validation Loss', fontweight='bold')
axes[3].set_title('Dropout Rate vs Performance', fontweight='bold')
axes[3].grid(True, alpha=0.3)

In [ ]:
# 5. Learning Rate vs Performance
lr_results = results_df.groupby('learning_rate')['val_loss'].agg(['mean', 'std']).reset_index()
axes[4].errorbar(lr_results['learning_rate'], lr_results['mean'], 
                yerr=lr_results['std'], marker='o', linewidth=2)
axes[4].set_xlabel('Learning Rate', fontweight='bold')
axes[4].set_ylabel('Validation Loss', fontweight='bold')
axes[4].set_title('Learning Rate vs Performance', fontweight='bold')
axes[4].set_xscale('log')
axes[4].grid(True, alpha=0.3)

In [ ]:
# 6. Batch Size vs Performance
batch_results = results_df.groupby('batch_size')['val_loss'].agg(['mean', 'std']).reset_index()
axes[5].errorbar(batch_results['batch_size'], batch_results['mean'], 
                yerr=batch_results['std'], marker='o', linewidth=2)
axes[5].set_xlabel('Batch Size', fontweight='bold')
axes[5].set_ylabel('Validation Loss', fontweight='bold')
axes[5].set_title('Batch Size vs Performance', fontweight='bold')
axes[5].grid(True, alpha=0.3)

In [ ]:
plt.tight_layout()
plt.savefig('gcn_grid_search_results.png', dpi=300, bbox_inches='tight')
plt.show()

# SAVE RESULTS

In [ ]:
# Save detailed results
results_df.to_csv('gcn_grid_search_results.csv', index=False)

In [ ]:
# Save best parameters
best_params_df = pd.DataFrame([best_params])
best_params_df['best_val_loss'] = best_score
best_params_df.to_csv('gcn_best_parameters.csv', index=False)

In [ ]:
print(f"\n✅ GCN Grid Search completed successfully!")
print(f"📁 Results saved:")
print(f"  📊 Detailed results: 'gcn_grid_search_results.csv'")
print(f"  🏆 Best parameters: 'gcn_best_parameters.csv'")
print(f"  📈 Visualizations: 'gcn_grid_search_results.png'")
print(f"  ⏱️ Total time: {total_time:.2f} seconds")